In [1]:
import json
import os
import torch
import sqlglot
from sqlglot.expressions import Table
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/hyeonjin/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 경로 설정 (사용자 디렉토리 구조 반영)
BASE_DIR = "../data/BIRD_dev"
DEV_JSON_PATH = os.path.join(BASE_DIR, "dev.json")
DEV_TABLES_PATH = os.path.join(BASE_DIR, "dev_tables.json")

In [3]:
# 2. 데이터 로드
with open(DEV_JSON_PATH, 'r', encoding='utf-8') as f:
    dev_data = json.load(f)

with open(DEV_TABLES_PATH, 'r', encoding='utf-8') as f:
    tables_data = json.load(f)

db_schema_map = {item['db_id']: item for item in tables_data}

In [4]:
from sqlglot.expressions import Column

# 3. Gold Column 추출기 (sqlglot 활용)
def extract_gold_columns(sql_query):
    """
    Gold SQL을 파싱하여 명시적으로 사용된 모든 '컬럼명'을 추출합니다.
    (테이블명 추출에서 컬럼명 추출로 변경됨)
    """
    try:
        parsed = sqlglot.parse_one(sql_query, read="sqlite")
        # AST 노드 중 'Column' 타입인 것들의 이름만 추출하여 소문자로 저장
        return set(node.name.lower() for node in parsed.find_all(Column))
    except Exception:
        # 파싱 실패 시 빈 집합 반환 (이 경우 프롬프트가 비어버릴 수 있으므로 주의)
        return set()

In [5]:
# 4. 스키마 프롬프트 생성기 (Column-Level 필터링 적용)
def build_schema_dot_format_column_level(db_id, used_columns=None):
    schema_info = db_schema_map.get(db_id)
    if not schema_info: return ""
    
    table_names = schema_info['table_names_original']
    column_names = schema_info['column_names_original']
    
    schema_lines = []
    
    # column_names 구조: [[table_idx, column_name], ...]
    for table_idx, col_name in column_names:
        if table_idx < 0: continue # 인덱스 -1은 '*' 이므로 제외
        
        t_name = table_names[table_idx]
        
        # [핵심 로직] Column-Level Oracle 모드일 경우:
        # 추출된 정답 컬럼 집합(used_columns)에 현재 컬럼명이 없으면 스킵합니다.
        if used_columns is not None:
            # sqlglot이 추출한 컬럼명과 BIRD 스키마의 컬럼명을 소문자로 비교
            if col_name.lower() not in used_columns:
                continue
                
        schema_lines.append(f"{t_name}.{col_name}")
        
    # 만약 파싱 에러나 컬럼 매칭 실패로 인해 schema_lines가 완전히 비어버린다면,
    # 해당 쿼리의 생성을 보조하기 위해 최소한의 Fallback으로 전체 스키마를 주거나 빈 문자열을 반환합니다.
    # 여기서는 엄격한 Upper Bound 테스트를 위해 빈 문자열을 허용합니다.
    return "\n".join(schema_lines)

In [6]:
# 5. 프롬프트 템플릿 (Llama-3.1 Instruct에 최적화된 System/User 분리)
def create_messages(schema_str, question, evidence):
    system_prompt = (
        "You are an expert SQL developer. Your task is to write a SQLite query based on the given schema and external knowledge. "
        "IMPORTANT: If a column name contains spaces or special characters, you MUST wrap it in backticks (e.g., `Column Name` or `Percent (%)`). "
        "Output ONLY the SQL query. Do not include markdown formatting like ```sql or any explanations."
    )
    user_prompt = f"### Schema (table.column):\n{schema_str}\n\n### External Knowledge:\n{evidence}\n\n### Question:\n{question}\n\n### SQL:"
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [7]:
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print("Loading tokenizer and model (This is safe for memory)...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# device_map="auto"가 가용한 여러 GPU에 모델을 안전하게 분산 배치합니다.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True # 시스템 RAM 스파이크 방지 핵심 옵션
)

# 패딩 토큰 설정 (Llama 3 특성)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading tokenizer and model (This is safe for memory)...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.43it/s]


In [8]:
def generate_sql(messages):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,      # Greedy Decoding
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # 입력 프롬프트 부분을 제외하고 생성된 텍스트만 추출
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

In [9]:
import threading
import sqlite3

DB_DIR = os.path.join(BASE_DIR, "dev_databases")

def execute_sql_on_db(db_path, sql_query, timeout_sec=3.0):
    """
    무한 루프(Cross Join) 방지를 위해 Thread를 이용한 엄격한 타임아웃을 적용합니다.
    """
    result = [None]
    exception = [None]

    def target():
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute(sql_query)
            result[0] = set(cursor.fetchall())
            conn.close()
        except Exception as e:
            exception[0] = str(e)

    thread = threading.Thread(target=target)
    thread.start()
    thread.join(timeout_sec)

    if thread.is_alive():
        return "Error: Query Timeout (Possible Cross Join)"
    
    if exception[0]:
        return exception[0]
        
    return result[0]

In [10]:
from tqdm import tqdm

print("\nStarting inference and EX evaluation...")

metrics = {
    "full_correct": 0,
    "gold_correct": 0,
    "total": 0
}

test_num = 1534

for i, item in tqdm(enumerate(dev_data[:test_num]), total=len(dev_data[:test_num])): # 실험 규모에 맞게 조절 (예: 50개)
    db_id = item['db_id']
    question = item['question']
    evidence = item['evidence']
    gold_sql = item['SQL']
    
    db_path = os.path.join(DB_DIR, db_id, f"{db_id}.sqlite")
    
    # --- 전처리 및 프롬프트 구성 ---
    gold_columns = extract_gold_columns(gold_sql)
    # Baseline (전체 스키마)
    schema_full = build_schema_dot_format_column_level(db_id, used_columns=None)
    # Oracle (정답 컬럼만 남긴 스키마)
    schema_gold = build_schema_dot_format_column_level(db_id, used_columns=gold_columns)
    
    msg_full = create_messages(schema_full, question, evidence)
    msg_gold = create_messages(schema_gold, question, evidence)
    
    # --- 모델 추론 ---
    pred_full = generate_sql(msg_full)
    pred_gold = generate_sql(msg_gold)
    
    # --- EX (Execution Accuracy) 평가 ---
    # 1. Gold SQL 실행 (Ground Truth 결과 확보)
    gold_result = execute_sql_on_db(db_path, gold_sql)
    
    # 2. Predicted SQL 실행
    pred_full_result = execute_sql_on_db(db_path, pred_full)
    pred_gold_result = execute_sql_on_db(db_path, pred_gold)
    
    # 3. 결과 비교 (Gold 결과가 정상적으로 나왔을 때만 평가)
    is_full_correct = (gold_result == pred_full_result) and not isinstance(gold_result, str)
    is_gold_correct = (gold_result == pred_gold_result) and not isinstance(gold_result, str)
    
    if is_full_correct: metrics["full_correct"] += 1
    if is_gold_correct: metrics["gold_correct"] += 1
    metrics["total"] += 1
    
    """
    # --- 결과 로깅 ---
    print(f"\n[{i+1}/{len(dev_data[:test_num])}] DB: {db_id}")
    print(f"Question: {question}") # 필요시 주석 해제
    
    # 오답 분석을 위한 로깅 (틀렸을 때만 SQL 출력)
    
    if not is_full_correct:
        print(f"❌ [Full Failed] Pred: {pred_full}")
        if isinstance(pred_full_result, str): print(f"   Error: {pred_full_result}")
            
    if not is_gold_correct:
        print(f"❌ [Gold Failed] Pred: {pred_gold}")
        if isinstance(pred_gold_result, str): print(f"   Error: {pred_gold_result}")
            
    if is_full_correct and is_gold_correct:
        print("✅ Both Correct")
    """

# --- 최종 결과 산출 ---
ex_full = (metrics["full_correct"] / metrics["total"]) * 100
ex_gold = (metrics["gold_correct"] / metrics["total"]) * 100
delta_ex = ex_gold - ex_full

print("\n" + "="*40)
print("🎯 EXPERIMENT RESULTS (Execution Accuracy)")
print("="*40)
print(f"Total Samples Tested: {metrics['total']}")
print(f"EX w/ Full Schema (Baseline) : {ex_full:.1f}%")
print(f"EX w/ Gold Schema (Oracle)   : {ex_gold:.1f}%")
print(f"Performance Gap ($\\Delta EX$)  : +{delta_ex:.1f}%p")
print("="*40)


Starting inference and EX evaluation for 10 samples...


100%|██████████| 1534/1534 [1:35:29<00:00,  3.73s/it]


🎯 EXPERIMENT RESULTS (Execution Accuracy)
Total Samples Tested: 1534
EX w/ Full Schema (Baseline) : 34.1%
EX w/ Gold Schema (Oracle)   : 41.5%
Performance Gap ($\Delta EX$)  : +7.4%p
